Two implementation of computing the sum of squares for 1 million numbers:

In [2]:
import numpy as np 
arr = np.arange(1_000_000, dtype=np.float64)


In [7]:
# Write the loop version using a Python for loop.
def square_sum_python_loop(arr):
    total=0.0
    for x in arr:
        total+=x*x
    return total
# Write the vectorized version using NumPy.
def square_sum_vectorized(arr):
    return np.sum(arr**2)


In [11]:
# Time both using %timeit and report the seedup factor.
%timeit square_sum_python_loop(arr)
%timeit square_sum_vectorized(arr)

107 ms ± 2.44 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
3.69 ms ± 300 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
# Now repeat with dtype=np.float32 on the vectorized version - does dtype affect speed?
arr2 = np.arange(1_000_000, dtype=np.float32)
%timeit square_sum_vectorized(arr2)

1.77 ms ± 109 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Yes, dtype affect speed. Higher the dtype size higher will be the time consumption.

Given this loop-base implementation:

In [22]:
np.random.seed(0)
data = np.random.randint(0, 100, size=(1000, 500))

def python_loop(data):
    result = []
    for row in data:
        count = 0
        for val in row:
            if val > 50:
                count += 1
        result.append(count)
    result = np.array(result)

In [16]:
# Rewrite this entirely without any Python loops - single line preferred.
def numpy_vectorized(data):
    return np.sum(data>50,axis=1)

In [18]:
# Time both version.
%timeit python_loop(data)
%timeit numpy_vectorized(data)

34.7 ms ± 1.54 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
454 μs ± 24.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
# Verify ouput match.
print("Matched?: ",np.array_equal(numpy_vectorized(data),result))

Matched?:  True


A large array and need to apply this piecewise logic:

In [57]:
arr = np.random.randint(-100, 100, size=2_000_000)

# Logic:
# if value < -50  → set to -50
# if value > 50   → set to 50
# else            → keep as is
print("Original sum: ",arr.sum())

Original sum:  -952307


In [67]:
# Write it first using np.vectorize() with a Python function.
import time as time
def manipulate(value):
    if value<-50:
        return -50
    elif value>50:
        return 50
    else:
        return value
fstart=time.time()
result = np.vectorize(manipulate)(arr)
fend=time.time()
print("Result sum: ",result.sum())

Result sum:  -481285


In [68]:
# Write the proper vectorized verson without np.vectorize()
vstart=time.time()
vresult=np.where(arr>50,50,np.where(arr<-50,-50,arr))
vend=time.time()
print("VResult sum: ",vresult.sum())

VResult sum:  -481285


In [69]:
# Time both - explain why one is faster despite both begin "NumPy".
print("Function vector time: ",fend-fstart)
print("Proper vecorized time: ",vend-vstart)


Function vector time:  0.31928372383117676
Proper vecorized time:  0.010689258575439453


Function version is slower because it is still uses python's loop under the hood. np.vectorize() just call the function iteratively which is python loop. Other version is faster because it uses numpy.

A (5000,4) dataset of sensor readings:

In [3]:
np.random.seed(1)
sensors = np.random.normal(50, 15, size=(5000, 4)).astype(np.float64)

In [4]:
# Compute row-wise mean using a Python loop over rows vs np.mean(axis=1). Time both approaches
def row_wise_mean(data):
    sum=0
    for i in data:
        sum+=i
    return sum/len(data)
def vectorized_mean(data):
    return np.mean(data,axis=1)

%timeit row_wise_mean(sensors)
%timeit vectorized_mean(sensors)

2.96 ms ± 208 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
73.8 μs ± 2.7 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [11]:
# Compare memory: convert sensors to float32 using astype(copy=False).
small_sensors=sensors.astype(np.float32,copy=False)
print("Original sensor memory use: ",sensors.nbytes)
print("Reduced sensor memory use: ",small_sensors.nbytes)
print("Memory saved: ",sensors.nbytes-small_sensors.nbytes)

Original sensor memory use:  160000
Reduced sensor memory use:  80000
Memory saved:  80000


Normalize each column of a large matrix to range [0,1]:

In [12]:
np.random.seed(5)
data = np.random.randint(0, 1000, size=(10_000, 6)).astype(float)

In [19]:
# Write the loop version: iterate over each column index, normalize it, store result.

# Normalized loop
normalized_loop=np.empty_like(data)
%timeit
for i in range(data.shape[1]):
    normalized_loop[:,i]=(data[:,i]-data[:,i].min())/(data[:,i].max()-data[:,i].min())

In [27]:
# Write the vectorized version using broadcasting and axis=0 min/max - single operation.

# Normalized vector
col_min=data.min(axis=0)
col_max=data.max(axis=0)
%timeit normalized_vec=(data-col_min)/(col_max-col_min)

205 μs ± 7.37 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [22]:
# --- Timing ---
print("Loop version timing:")
%timeit for i in range(data.shape[1]): (data[:, i] - data[:, i].min()) / (data[:, i].max() - data[:, i].min())

print("Vectorized version timing:")
%timeit (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0))

Loop version timing:
262 μs ± 13.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Vectorized version timing:
971 μs ± 22.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


- The loop version took less time, but only because it wasn’t storing results. The vectorized version did the full normalization and therefore looked slower.

In [ ]:
# Fair timing
# Proper loop timing
print("Loop version timing:")
%timeit normalized_loop = np.empty_like(data); \
    [normalized_loop.__setitem__((slice(None), i), (data[:, i] - data[:, i].min()) / (data[:, i].max() - data[:, i].min())) \
     for i in range(data.shape[1])]

# Proper vectorized timing
print("Vectorized version timing:")
%timeit (data - col_min) / (col_max - col_min)

Loop version timing:
343 μs ± 20.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Vectorized version timing:
194 μs ± 2.68 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


* The vectorized version is faster in this test — about 1.8× quicker than the loop version.
* This confirms the general expectation: vectorization usually outperforms explicit Python loops, especially as data size grows.

In [28]:
# Verifty both produce identical results.
print(np.allclose(normalized_loop,normalized_vec))


True


* Both methods produce identical normalized results (np.allclose(normalized_loop, normalized_vec) returns True).

Monthly transaction data for 1000 customers over 24 months:

In [29]:
np.random.seed(9)
transactions = np.random.randint(0, 5000, size=(1000, 24))

In [30]:
# Compute cumulative spend per customer across months.
cum_spend=np.cumsum(transactions,axis=1)
print(cum_spend.shape)

(1000, 24)


In [38]:
# From the result, extract only customers whose total spend exceeds 60,000.
total_spend=cum_spend[:,-1]
mask=total_spend<60_000
high_spend_customer=cum_spend[mask]
print("First 10 samples: \n",high_spend_customer[:10,:])

First 10 samples: 
 [[ 3281  3571  3715  4711  6417  7644 12599 13527 17321 20813 23758 26682
  30353 32099 34076 39025 40516 43938 45294 46713 49570 52708 55882 59935]
 [ 2517  3402  5716 10055 10077 13946 17531 20051 21611 25041 26530 27689
  30675 30675 35456 39585 42256 42954 47706 49285 50352 51921 53941 56110]
 [ 3714  5675  8642 12390 14328 14591 17058 21395 22083 23660 27995 29935
  32511 33346 33885 37219 37687 38719 41352 44118 44326 45683 49641 53677]
 [ 2864  3577  5558  7846  8652 12954 16864 17157 19697 21003 23041 23546
  25331 28689 33308 34495 36403 40414 42474 42610 43719 46511 47961 48487]
 [ 4340  4436  5835  8240  9635 10363 11531 11693 14371 15906 17582 18034
  20143 22029 24259 27417 32228 33784 37122 40598 44985 47351 48488 52275]
 [ 1871  3558  6627  9648 11267 12228 15145 16968 21371 23249 24936 25171
  26184 29331 31104 34000 34521 36510 41244 41320 44728 46573 48508 53071]
 [ 4366  6391  7903 10776 11505 11925 14932 15099 15564 19103 19527 22311
  23636 2585

In [44]:
# Report how many customers qualify and their average total spend.
print("Qualified customer count: ",high_spend_customer.shape[0])
print(f"Average total spend of qualified customers: ${np.mean(high_spend_customer[:,-1]):.2f}")

Qualified customer count:  518
Average total spend of qualified customers: $54285.10


Two approaches for computing the dot product of two large matrices:

In [62]:
np.random.seed(2)
A = np.random.rand(500, 300).astype(np.float64)
B = np.random.rand(300, 400).astype(np.float64)

In [63]:
# Time A @ B with float64
result64=A @ B
%timeit A @ B
print(result64.nbytes)

1.93 ms ± 124 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
1600000


In [64]:
# Convert both to float32 using in-place-style and time again.
A=A.astype(np.float32,copy=False) 
B=B.astype(np.float32,copy=False)
result32=A @ B 
print(result32.nbytes)
%timeit A @ B 

800000
774 μs ± 147 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [65]:
# Verify results are close.
print("Result closeness: ",np.allclose(result64,result32,atol=1e-3))

Result closeness:  True


In [66]:
# Report the memory difference between float64 and float32 versions of A and B combined.
print("Memory difference between float64 and float32 version: ",result64.nbytes-result32.nbytes)

Memory difference between float64 and float32 version:  800000
